# LLM-from-Scratch Playground

Interactive notebook for experimenting with the trained model.

In [ ]:
import os
import torch
from config import get_config
from data.tokenizer import BPETokenizer
from model.transformer import Transformer
from utils.training import CheckpointManager
from utils.sampling import generate_text

# Choose configuration
config = get_config("laptop")  # or "cloud"
device = "cuda" if torch.cuda.is_available() else "cpu"
config.device = device
print(f"Using device: {device}")

## Load Tokenizer and Model

In [ ]:
# Load tokenizer
tokenizer = BPETokenizer(vocab_size=config.vocab_size)
tokenizer.load(os.path.join(config.data_dir, "tokenizer"))

# Create model
model = Transformer(config).to(device)

# Load latest checkpoint
checkpoint_manager = CheckpointManager(config.checkpoint_dir)
checkpoint_files = sorted(
    [f for f in os.listdir(config.checkpoint_dir) if f.endswith(".pt")],
    key=lambda x: int(x.split("_")[-1].replace(".pt", ""))
)

if checkpoint_files:
    latest = os.path.join(config.checkpoint_dir, checkpoint_files[-1])
    checkpoint_manager.load(latest, model, device=device)
    print(f"Loaded checkpoint: {latest}")
else:
    print("No checkpoint found. Using randomly initialized model.")

## Interactive Text Generation

In [ ]:
prompt = "The future of artificial intelligence is"

output = generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    top_k=40,
    top_p=1.0,
    device=device,
)

print("Prompt:", prompt)
print("-" * 60)
print(output)
print("-" * 60)

## Experiment with Sampling Parameters

In [ ]:
prompt = "In the beginning"

for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\n=== Temperature: {temp} ===")
    out = generate_text(
        model, tokenizer, prompt,
        max_new_tokens=100, temperature=temp, top_k=40, device=device
    )
    print(out[len(prompt):])  # Print only generated part

## Analyze Model Architecture

In [ ]:
# Model summary
print(f"Total parameters: {model.get_num_params():,}")
print(f"Model size: {model.get_num_params() * 4 / 1024 / 1024:.2f} MB (float32)")

# Parameter breakdown
for name, param in model.named_parameters():
    print(f"{name:40s} {str(tuple(param.shape)):20s} {param.numel():>10,}")

## Visualize Training Loss

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = os.path.join(config.log_dir, f"{config.vocab_size}_training_log.csv")
if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    ax.plot(df["step"], df["train_loss"], label="Train Loss")
    
    val_df = df.dropna(subset=["val_loss"])
    if not val_df.empty:
        ax.plot(val_df["step"], val_df["val_loss"], label="Val Loss", linestyle="--")
    
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Curve")
    ax.legend()
    ax.grid(True)
    plt.show()
else:
    print("No training log found.")